# T10 It2: inlier-scale robustness of the judge

T9 showed the judge answers wrong places when every candidate's inlier signals rise together (x1.25 -> 2-3 wrong on NUDT 707), mostly
through the 12-inlier override. Variants (block `train/checks/scale_robust.py`), each trained on NUDT design sets only, cut by Learn-then-Test
(alpha 0.005, floor 0.9) on the training sets (for augmented variants: on every scaled copy, so the cut must hold across the range):

- V0 current judge + 12-inlier override (the T8 recommendation)
- V1 no override
- V2 relative inliers: rli = log((inliers + 1) / (running far-candidate inliers + 1)), likewise n22; the far (> 4 sigma) candidates are
  almost all wrong places, so their level measures this camera / map / matcher without labels (running mean per run, dev prior, weight 30)
- V3 scale augmentation: train on copies of every set with inlier signals x0.5, 0.75, 1, 1.5, 2
- V4 no override + safety switch: photos whose running far-inlier level is outside the dev range (+-25 %) use the strict mode (cut 0.98)
- V5 relative + augmentation

**Test:** leave one NUDT design set out (train on the other four) and official TRY (train on all five), with every inlier-count signal of
the held-out set scaled x0.5, 0.75, 1, 1.25, 1.5, 2. PC (M = 10) and Jetson (M = 5, context from the 5 matched candidates). **A variant
counts only with 0 wrong answers at every scale on every held-out set**; the score kept is reported against x1.

In [1]:
import os, sys, time
os.environ["OMP_NUM_THREADS"] = "1"                    # 12 processes below, one thread each
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.holdout import NUDT_DESIGN, load_t8_table
from train.checks.eval_tools import manifest, save_manifest
RES = ROOT / "tests/Test_Iteration_10/results"
T = load_t8_table(ROOT, locked=True)            # NUDT design + official TRY only
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
F = cj.FEATURES_CTX; INC = {"li", "z", "hull", "cover9"}
FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]; INCR = {"rli", "z", "hull", "cover9"}
VARIANTS = {   # name: (features, increasing, augment, relative, override, safety)
    "V0 current + override 12": (F, INC, None, False, 12, False),
    "V1 no override": (F, INC, None, False, None, False),
    "V2 relative inliers": (FR, INCR, None, True, None, False),
    "V3 scale augmentation": (F, INC, AUG, False, None, False),
    "V4 no override + safety switch": (F, INC, None, False, None, True),
    "V5 relative + augmentation": (FR, INCR, AUG, True, None, False)}

In [2]:
def run(var_name, ctx_m, held_sets, train_sets):
    feats, inc, aug, rel, override, safety = VARIANTS[var_name]
    train = T[T.data.isin(train_sets)]
    if ctx_m: train = cj.add_context(train, m=ctx_m)
    lv = sr.far_levels(train, ctx_m); prior = dict(far_inl=lv.far_inl.mean(), far_n22=lv.far_n22.mean())
    tr = (lambda t: sr.add_relative(t, prior, 30, ctx_m)) if rel else None
    model, p_oof, stacked = sr.fit_judge(train, train_sets, feats, inc, aug, tr)
    kw = dict(min_choice=3, wide_tau=0.95, override=override)
    if ctx_m: kw.update(jetson=True, m=ctx_m)
    names = stacked.data + "_x" + stacked.scale_factor.astype(str)
    tau = cj.ltt_tau(cj.scan_tau(stacked.assign(data=names), p_oof, cj.TAUS, **kw), list(names.unique()), 0.005)
    tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
    fin = sr.running_level(lv, prior).groupby("data").run_inl.last()
    low, high = fin.min() * 0.75, fin.max() * 1.25
    rows = []
    for f in SCALES:
        h = sr.add_relative(sr.scale_signals(T[T.data.isin(held_sets)], f, m=ctx_m), prior, 30, ctx_m)
        p = sr.predict(model, h, feats)
        strict_share = 0.0
        if safety:
            strict = sr.safety_mode(h, low, high); p = np.where(strict & (p < 0.98), 0.0, p)
            strict_share = float(h.assign(s=strict).groupby(["data", "photo"]).s.first().mean())
        s = cj.summary(cj.decide(h, p, tau, **kw))
        for d in s.index:
            rows.append(dict(variant=var_name, var="jetson" if ctx_m else "pc", held=d, scale=f, tau=tau, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]),
                             answered=s.loc[d, "answered"], queries=s.loc[d, "queries"], strict_share=strict_share))
    return rows

from joblib import Parallel, delayed
jobs = [(v, m, [h], [s for s in NUDT_DESIGN if s != h]) for v in VARIANTS for m in (None, 5) for h in NUDT_DESIGN]
jobs += [(v, m, ["official_try"], NUDT_DESIGN) for v in VARIANTS for m in (None, 5)]
t0 = time.time()
SWEEP = pd.DataFrame([r for rr in Parallel(n_jobs=int(os.environ.get("JOBS", "4")))(delayed(run)(*j) for j in jobs) for r in rr])
SWEEP.to_csv(RES / "it2_scale_sweep.csv", index=False)
save_manifest(RES / "it2_scale_sweep.csv", manifest("pc", cut="Learn-then-Test alpha 0.005 per variant, floor 0.9", wide_cut=0.95, override="V0: 12, others none",
              judge_features="FEATURES_CTX or relative (rli, rln22)", judge_trained_on="NUDT design sets, leave one out", note="M = 10 (pc) and M = 5 (jetson) rows"))
print("%d runs, %.0f s" % (len(jobs), time.time() - t0))

/home/palash/Ideaforge/GeoLoc/.venv-geoloc/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


72 runs, 2146 s


## Wrong answers summed over the held-out sets, per scale

In [3]:
SWEEP.pivot_table(index=["var", "variant"], columns="scale", values="wrong", aggfunc="sum").astype(int)

scale                                  0.50  0.75  1.00  1.25  1.50  2.00
var    variant                                                           
jetson V0 current + override 12           0     1     1     2     5    16
       V1 no override                     0     1     1     2     3     5
       V2 relative inliers                0     1     1     1     3     6
       V3 scale augmentation              1     2     2     2     2     2
       V4 no override + safety switch     0     1     1     2     1     1
       V5 relative + augmentation         1     1     1     2     2     2
pc     V0 current + override 12           0     0     1     5    15    31
       V1 no override                     0     0     1     3     5     6
       V2 relative inliers                0     1     1     2     5     6
       V3 scale augmentation              1     1     2     2     1     2
       V4 no override + safety switch     0     0     1     3     3     3
       V5 relative + augmentation         1     1     1     1     1     1

## Score (mean over the held-out NUDT sets, and official TRY) per scale

In [4]:
S2 = SWEEP.assign(group=np.where(SWEEP.held == "official_try", "official_try", "NUDT (mean of 5)"))
S2.pivot_table(index=["var", "variant", "group"], columns="scale", values="score", aggfunc="mean").round(2)

scale                                                    0.50   0.75   1.00   1.25   1.50   2.00
var    variant                        group                                                     
jetson V0 current + override 12       NUDT (mean of 5)  54.13  54.86  55.29  55.39  55.42  54.40
                                      official_try      51.76  52.22  52.86  53.19  53.51  54.68
       V1 no override                 NUDT (mean of 5)  54.05  54.86  55.29  55.25  55.38  55.40
                                      official_try      51.76  52.22  52.86  53.19  53.51  54.68
       V2 relative inliers            NUDT (mean of 5)  54.20  54.82  55.14  55.30  55.23  55.06
                                      official_try      52.09  52.86  52.86  54.49  54.49  54.68
       V3 scale augmentation          NUDT (mean of 5)  54.92  55.11  55.21  55.27  55.33  55.45
                                      official_try      52.09  52.86  53.06  53.06  53.06  53.38
       V4 no override + safety switch NUDT (mean of 5)  53.40  54.86  55.29  55.13  55.13  55.06
                                      official_try      49.62  51.05  52.22  52.54  52.54  52.86
       V5 relative + augmentation     NUDT (mean of 5)  54.92  55.26  55.43  55.30  55.39  55.41
                                      official_try      52.09  52.86  53.38  53.38  53.38  54.68
pc     V0 current + override 12       NUDT (mean of 5)  55.28  56.16  56.25  56.09  55.34  53.66
                                      official_try      51.76  53.51  54.03  54.03  54.68  55.98
       V1 no override                 NUDT (mean of 5)  55.24  56.12  56.17  56.13  56.15  56.07
                                      official_try      51.76  53.51  54.03  54.03  54.68  55.98
       V2 relative inliers            NUDT (mean of 5)  55.37  56.10  56.39  56.39  56.27  56.14
                                      official_try      52.74  53.71  54.03  54.55  55.65  55.98
       V3 scale augmentation          NUDT (mean of 5)  55.69  56.07  56.03  56.09  56.25  56.17
                                      official_try      53.06  54.36  54.55  54.55  54.55  54.55
       V4 no override + safety switch NUDT (mean of 5)  54.59  56.12  56.17  55.97  55.67  55.71
                                      official_try      50.27  51.70  52.54  53.84  54.03  54.81
       V5 relative + augmentation     NUDT (mean of 5)  55.79  56.16  56.25  56.35  56.35  56.39
                                      official_try      53.06  54.36  54.55  54.55  54.55  55.65

In [5]:
x1 = S2[S2.scale == 1.0].groupby(["var", "variant", "group"]).score.mean()
ok = SWEEP.groupby(["var", "variant"]).wrong.sum().eq(0).rename("0 wrong at every scale")
kept = S2.groupby(["var", "variant", "group"]).score.mean().rename("mean over scales")
pd.concat([x1.rename("score at x1"), kept], axis=1).join(ok, on=["var", "variant"]).round(2)

score at x1  mean over scales  0 wrong at every scale
var    variant                        group                                                                  
jetson V0 current + override 12       NUDT (mean of 5)        55.29             54.92                   False
                                      official_try            52.86             53.04                   False
       V1 no override                 NUDT (mean of 5)        55.29             55.04                   False
                                      official_try            52.86             53.04                   False
       V2 relative inliers            NUDT (mean of 5)        55.14             54.96                   False
                                      official_try            52.86             53.58                   False
       V3 scale augmentation          NUDT (mean of 5)        55.21             55.21                   False
                                      official_try            53.06             52.92                   False
       V4 no override + safety switch NUDT (mean of 5)        55.29             54.81                   False
                                      official_try            52.22             51.81                   False
       V5 relative + augmentation     NUDT (mean of 5)        55.43             55.28                   False
                                      official_try            53.38             53.30                   False
pc     V0 current + override 12       NUDT (mean of 5)        56.25             55.46                   False
                                      official_try            54.03             54.00                   False
       V1 no override                 NUDT (mean of 5)        56.17             55.98                   False
                                      official_try            54.03             54.00                   False
       V2 relative inliers            NUDT (mean of 5)        56.39             56.11                   False
                                      official_try            54.03             54.44                   False
       V3 scale augmentation          NUDT (mean of 5)        56.03             56.05                   False
                                      official_try            54.55             54.27                   False
       V4 no override + safety switch NUDT (mean of 5)        56.17             55.70                   False
                                      official_try            52.54             52.86                   False
       V5 relative + augmentation     NUDT (mean of 5)        56.25             56.22                   False
                                      official_try            54.55             54.45                   False